# 🇪🇸 Opposition Analysis: Spain's Build-up from Goal Kicks
### UEFA Euro 2024 — Prepared for Italy Coaching Staff

---

**Objective:** Analyse how Spain builds up from goalkeeper distributions (goal kicks) across all 7 matches at Euro 2024, identify patterns, vulnerabilities and provide tactical recommendations for Italy.

**Data Source:** StatsBomb open event data — 187,858 events across 51 Euro 2024 matches.

**Methodology at a glance:**

1. **Identify** every Spain goal kick (pass with `pass_type == "Goal Kick"`)
2. **Classify** distribution type as **Short** (receiver in zones 1–6, own third) or **Long** (zones 7–18)
3. **Track** the first receiver and the subsequent event chain
4. **Classify outcome** using a 6-tier system:
   - ✅ **P1** — Established possession (kept ball ≥15s in own half)
   - ✅ **P2** — Reached final third (kept ball ≥15s, progressed past x=80)
   - ✅ **P3** — Created a shot (retained possession → shot attempt)
   - ❌ **N1** — Possession lost (opponent recovered, no danger created)
   - ❌ **N2** — Box entry conceded (opponent entered penalty area)
   - ❌ **N3** — Shot conceded (opponent produced a shot after recovery)
5. **Visualise** distribution patterns, target zones, receivers and event chains
6. **Recommend** tactical actions for Italy's pressing plan

In [2]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Imports & Configuration
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ast
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──
DATA_DIR = '../AnalisisDeJuego_Europea_Marzo26'
ALL_EVENTS = f'{DATA_DIR}/SB_Euro24_allevents.xlsx'
MATCH_DETAILS = f'{DATA_DIR}/SB_Euro24_matchdetails.xlsx'

# ── Analysis Configuration ──
TEAM = 'Spain'
POSSESSION_WINDOW_SEC = 15.0   # seconds to track after first reception

# ── StatsBomb pitch dimensions ──
SB_X_MAX = 120.0
SB_Y_MAX = 80.0

# ── 18-Zone Grid (adapted to StatsBomb 120×80) ──
ROW_WIDTH = SB_X_MAX / 6    # 20.0
COL_WIDTH = SB_Y_MAX / 3    # 26.67

# Zone thresholds
SHORT_ZONES = set(range(1, 7))    # Z1–Z6  (defensive third: x ≤ 40)
LONG_ZONES  = set(range(7, 19))   # Z7–Z18 (middle + attacking third)

FINAL_THIRD_X = SB_X_MAX / 6 * 4  # 80.0
BOX_X = SB_X_MAX / 6 * 5          # 100.0

# Zone grid edges (StatsBomb scale)
X_EDGES = [0, 20, 40, 60, 80, 100, 120]
Y_EDGES = [0, 26.67, 53.33, 80]

# Non-play events to skip
NON_PLAY_EVENTS = frozenset({
    'starting xi', 'half start', 'half end', 'substitution',
    'player on', 'player off', 'tactical shift', 'referee ball-drop',
    'injury stoppage', 'bad behaviour', 'shield',
})

# Shot events
SHOT_EVENTS = frozenset({'shot'})

# Contested events (opponent action that doesn't necessarily mean possession change)
CONTESTED_EVENTS = frozenset({
    'duel', 'block', 'interception', 'ball recovery',
    'clearance', 'pressure', '50/50',
})

# Team loss events
TEAM_LOSS_EVENTS = frozenset({'dispossessed', 'error', 'miscontrol'})

# Team constructive events
TEAM_CONSTRUCTIVE = frozenset({
    'pass', 'ball receipt*', 'carry', 'dribble', 'clearance',
    'shot', 'foul won', 'goal keeper',
})

print('✅ Configuration loaded')
print(f'   Team: {TEAM}')
print(f'   StatsBomb pitch: {SB_X_MAX}×{SB_Y_MAX}')
print(f'   Zone grid: 6 rows × 3 cols = 18 zones')
print(f'   Row width: {ROW_WIDTH}, Col width: {COL_WIDTH:.2f}')

✅ Configuration loaded
   Team: Spain
   StatsBomb pitch: 120.0×80.0
   Zone grid: 6 rows × 3 cols = 18 zones
   Row width: 20.0, Col width: 26.67


---
## 1. How We Define the 18-Zone Grid

The pitch is divided into **18 zones** (6 rows × 3 columns) on StatsBomb's 120×80 coordinate system. This lets us pinpoint exactly where Spain's first receiver collects the ball.

| Row | X range | Meaning |
|-----|---------|---------|
| 1–2 | 0–40 | **Own third** (Short distribution) |
| 3–4 | 40–80 | **Middle third** |
| 5–6 | 80–120 | **Final third / Box** |

**Columns:** Left (y 0–27) · Centre (y 27–53) · Right (y 53–80)

If the first receiver lands in **zones 1–6** → **Short**. Zones **7–18** → **Long**.

In [3]:
# ══════════════════════════════════════════════════════════════
# CELL 2 — Load & Prepare Data
# ══════════════════════════════════════════════════════════════

# Load all events
df_all = pd.read_excel(ALL_EVENTS)
df_matches = pd.read_excel(MATCH_DETAILS)

# Parse location columns (StatsBomb stores as string '[x, y]')
def parse_location(val):
    """Parse StatsBomb location string '[x, y]' → (x, y) or (None, None)."""
    if pd.isna(val):
        return None, None
    try:
        if isinstance(val, str):
            coords = ast.literal_eval(val)
        else:
            coords = val
        return float(coords[0]), float(coords[1])
    except:
        return None, None

# Parse locations into separate x, y columns
df_all[['loc_x', 'loc_y']] = df_all['location'].apply(
    lambda v: pd.Series(parse_location(v))
)
df_all[['pass_end_x', 'pass_end_y']] = df_all['pass_end_location'].apply(
    lambda v: pd.Series(parse_location(v))
)

# Sort events correctly within each match
df_all = df_all.sort_values(['match_id', 'period', 'minute', 'second', 'index']).reset_index(drop=True)

# Parse timestamp for elapsed time calculations
def parse_timestamp_to_seconds(ts, minute, period):
    """Convert StatsBomb timestamp 'HH:MM:SS.mmm' to total seconds within the period."""
    if pd.notna(ts):
        try:
            parts = str(ts).split(':')
            if len(parts) == 3:
                # HH:MM:SS.mmm → total seconds
                return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
            elif len(parts) == 2:
                return int(parts[0]) * 60 + float(parts[1])
        except:
            pass
    return float(minute) * 60 if pd.notna(minute) else 0.0

df_all['time_seconds'] = df_all.apply(
    lambda r: parse_timestamp_to_seconds(r['timestamp'], r['minute'], r['period']), axis=1
)

# Spain matches info
spain_matches = df_matches[
    (df_matches['home_team'] == TEAM) | (df_matches['away_team'] == TEAM)
].copy()
spain_matches['opponent'] = spain_matches.apply(
    lambda r: r['away_team'] if r['home_team'] == TEAM else r['home_team'], axis=1
)


print(f'✅ Loaded {len(df_all):,} events across {df_all["match_id"].nunique()} matches')
print(f'\n📋 Spain matches at Euro 2024:')
for _, m in spain_matches.iterrows():
    print(f"   {m['competition_stage']:20s} | Spain vs {m['opponent']:15s} | {m['home_score']}-{m['away_score']}")

✅ Loaded 187,858 events across 51 matches

📋 Spain matches at Euro 2024:
   Final                | Spain vs England         | 2-1
   Semi-finals          | Spain vs France          | 2-1
   Quarter-finals       | Spain vs Germany         | 2-1
   Round of 16          | Spain vs Georgia         | 4-1
   Group Stage          | Spain vs Albania         | 0-1
   Group Stage          | Spain vs Italy           | 1-0
   Group Stage          | Spain vs Croatia         | 3-0


In [4]:
# ══════════════════════════════════════════════════════════════
# CELL 3 — Core Analysis Functions (StatsBomb-adapted)
# ══════════════════════════════════════════════════════════════

def xy_to_zone(x: float, y: float) -> int:
    """
    Convert StatsBomb (x, y) → zone 1–18.
    
    StatsBomb: x ∈ [0,120], y ∈ [0,80]
    Grid: 6 rows (x-axis) × 3 cols (y-axis)
    
    Zone layout (team attacks left → right):
      Row 6 (x 100–120): Z16 Z17 Z18   ← opponent box
      Row 5 (x  80–100): Z13 Z14 Z15
      Row 4 (x  60–80):  Z10 Z11 Z12
      Row 3 (x  40–60):   Z7  Z8  Z9
      Row 2 (x  20–40):   Z4  Z5  Z6
      Row 1 (x   0–20):   Z1  Z2  Z3   ← own box
    
    Columns: Left (y 0–26.67) | Centre (y 26.67–53.33) | Right (y 53.33–80)
    """
    x = max(0.0, min(SB_X_MAX, float(x)))
    y = max(0.0, min(SB_Y_MAX, float(y)))
    row = min(int(x / ROW_WIDTH), 5)
    col = min(int(y / COL_WIDTH), 2)
    return row * 3 + col + 1


def _is_play_event(event_type: str) -> bool:
    """Return True if event is an on-pitch action."""
    return event_type.lower() not in NON_PLAY_EVENTS and event_type != ''


def _elapsed_seconds(ref_row, row):
    """Seconds elapsed between two events."""
    if ref_row['period'] != row['period']:
        return POSSESSION_WINDOW_SEC + 1
    return row['time_seconds'] - ref_row['time_seconds']


def _find_first_receiver(gk_idx, match_df):
    """
    After a goal kick at gk_idx, find the first teammate touch.
    In StatsBomb data, the first Ball Receipt* by a teammate is the receiver.
    Skips Carry (same player continuation) and Pressure (defensive action).
    """
    for j in range(gk_idx + 1, min(gk_idx + 25, len(match_df))):
        row = match_df.iloc[j]
        et = str(row.get('type', '')).strip()
        
        if not _is_play_event(et):
            continue
        
        # Skip Carry events (they follow Ball Receipt, same player)
        if et == 'Carry':
            continue
        # Skip Pressure (opponent defensive action, not a possession change)
        if et == 'Pressure':
            continue
        
        # Same team → first receiver found
        if row['team'] == TEAM:
            rx, ry = row['loc_x'], row['loc_y']
            if pd.notna(rx) and pd.notna(ry):
                zone = xy_to_zone(rx, ry)
            else:
                zone = 0
            return {
                'iloc': j,
                'player': str(row.get('player', '?')),
                'x': rx, 'y': ry,
                'zone': zone,
                'event_type': et,
            }
        else:
            # Opponent touched first → intercepted
            return None
    return None


def _event_record(row):
    """Build a chain event dict from a row."""
    return {
        'player': str(row.get('player', '?')),
        'event_type': str(row.get('type', '')).strip(),
        'x': float(row['loc_x']) if pd.notna(row.get('loc_x')) else None,
        'y': float(row['loc_y']) if pd.notna(row.get('loc_y')) else None,
        'minute': int(row.get('minute', 0)),
        'second': int(row.get('second', 0)),
        'is_team': row.get('team') == TEAM,
    }


def _track_opponent_danger(start_idx, match_df):
    """
    After possession loss, check subsequent opponent events for danger level.
    Returns 'N1' (simple loss), 'N2' (box entry), or 'N3' (shot conceded).
    """
    entered_box = False
    for j in range(start_idx, min(start_idx + 40, len(match_df))):
        row = match_df.iloc[j]
        et = str(row.get('type', '')).strip()
        if not _is_play_event(et.lower()):
            continue
        # If possession changes back to Spain, stop tracking
        if row.get('possession_team') == TEAM and row['team'] == TEAM:
            break
        if row['team'] != TEAM:
            if et == 'Shot':
                return 'N3'
            ox = row.get('loc_x')
            if pd.notna(ox) and float(ox) >= BOX_X:
                entered_box = True
    return 'N2' if entered_box else 'N1'


def _extract_chain_and_outcome(gk_idx, recv_idx, match_df):
    """
    Extract event chain and classify outcome using StatsBomb's possession_team
    for reliable possession tracking within a 15-second window.
    
    Key StatsBomb adaptations:
    - Uses possession_team to detect genuine possession changes
    - Skips Pressure events (defensive, not possession changes)
    - Properly handles Ball Receipt* / Carry chains
    
    Returns (chain, outcome, granular_outcome)
    """
    chain = []
    gk_row = match_df.iloc[gk_idx]
    chain.append(_event_record(gk_row))
    
    # ── No receiver (intercepted immediately) ──
    if recv_idx is None:
        pass_outcome = str(gk_row.get('pass_outcome', '')).strip()
        if pass_outcome in ('Incomplete', 'Out'):
            return chain, 'negative', 'N1'
        # Scan for what happens next
        for j in range(gk_idx + 1, min(gk_idx + 30, len(match_df))):
            row = match_df.iloc[j]
            et = str(row.get('type', '')).strip()
            if not _is_play_event(et):
                continue
            if et in ('Pressure', 'Carry'):
                continue
            chain.append(_event_record(row))
            if row['team'] != TEAM:
                neg_level = _track_opponent_danger(j, match_df)
                return chain, 'negative', neg_level
            break
        return chain, 'negative', 'N1'
    
    # ── Have a receiver — track possession within 15s window ──
    ref_row = match_df.iloc[recv_idx]
    if recv_idx != gk_idx:
        chain.append(_event_record(ref_row))
    
    reached_final_third = False
    rx = ref_row.get('loc_x')
    if pd.notna(rx) and float(rx) >= FINAL_THIRD_X:
        reached_final_third = True
    
    # Track consecutive opponent constructive actions to detect de-facto
    # possession even when StatsBomb's possession_team hasn't updated yet.
    OPPONENT_CONSTRUCTIVE = frozenset({'Pass', 'Ball Receipt*', 'Carry', 'Dribble', 'Shot'})
    opp_consecutive = 0  # how many consecutive constructive opponent actions
    
    j = recv_idx + 1
    while j < len(match_df):
        row = match_df.iloc[j]
        elapsed = _elapsed_seconds(ref_row, row)
        et = str(row.get('type', '')).strip()
        et_lower = et.lower()
        
        if not _is_play_event(et_lower):
            j += 1
            continue
        
        is_team = row['team'] == TEAM
        poss_team = row.get('possession_team')
        
        # ── Team action ──
        if is_team:
            chain.append(_event_record(row))
            opp_consecutive = 0  # reset opponent streak
            
            # Shot by the team → P3
            if et == 'Shot':
                return chain, 'positive', 'P3'
            
            # Check for final-third entry
            ex = row.get('loc_x')
            if pd.notna(ex) and float(ex) >= FINAL_THIRD_X:
                reached_final_third = True
            
            # If 15s elapsed and team still has the ball → positive
            if elapsed >= POSSESSION_WINDOW_SEC:
                if reached_final_third:
                    return chain, 'positive', 'P2'
                return chain, 'positive', 'P1'
            
            j += 1
            continue
        
        # ── Opponent event ──
        # Skip Pressure (defensive action, not possession change)
        if et == 'Pressure':
            j += 1
            continue
        
        chain.append(_event_record(row))
        
        # Track opponent constructive actions
        if et in OPPONENT_CONSTRUCTIVE:
            opp_consecutive += 1
        
        # Foul committed by opponent:
        # Only positive if opponent did NOT already have constructive possession.
        # If they made a Pass or Receipt before fouling, they had the ball → negative.
        if et == 'Foul Committed':
            if opp_consecutive == 0:
                # Foul on Spain player without opponent having the ball → positive
                if reached_final_third:
                    return chain, 'positive', 'P2'
                return chain, 'positive', 'P1'
            else:
                # Opponent had the ball, then fouled → treat as negative
                neg_level = _track_opponent_danger(j, match_df)
                return chain, 'negative', neg_level
        
        # Use StatsBomb's possession_team for reliable possession detection
        if poss_team != TEAM:
            neg_level = _track_opponent_danger(j, match_df)
            return chain, 'negative', neg_level
        
        # Also detect de-facto possession loss: if opponent has made ≥2
        # consecutive constructive actions, they effectively have the ball
        # even if possession_team hasn't updated yet.
        if opp_consecutive >= 2:
            neg_level = _track_opponent_danger(j, match_df)
            return chain, 'negative', neg_level
        
        j += 1
        continue
    
    # End of data with team in possession
    if reached_final_third:
        return chain, 'positive', 'P2'
    return chain, 'positive', 'P1'


print('✅ Analysis functions defined (StatsBomb-adapted)')
print('   Key adaptations:')
print('   - Uses possession_team for reliable possession tracking')
print('   - Skips Pressure events (defensive action, not possession change)')
print('   - Handles Ball Receipt* / Carry event chains correctly')
print('   - Tracks opponent danger after possession loss (N1/N2/N3)')
print('   - Detects de-facto possession loss (≥2 consecutive opponent actions)')
print('   - Opponent fouls only positive if they did not already have the ball')

✅ Analysis functions defined (StatsBomb-adapted)
   Key adaptations:
   - Uses possession_team for reliable possession tracking
   - Skips Pressure events (defensive action, not possession change)
   - Handles Ball Receipt* / Carry event chains correctly
   - Tracks opponent danger after possession loss (N1/N2/N3)
   - Detects de-facto possession loss (≥2 consecutive opponent actions)
   - Opponent fouls only positive if they did not already have the ball


In [5]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Run Analysis Across All Spain Matches
# ══════════════════════════════════════════════════════════════

spain_match_ids = spain_matches['match_id'].tolist()
all_distributions = []

for match_id in spain_match_ids:
    match_df = df_all[df_all['match_id'] == match_id].copy()
    match_df = match_df.sort_values(['period', 'minute', 'second', 'index']).reset_index(drop=True)
    
    # Get opponent name
    match_info = spain_matches[spain_matches['match_id'] == match_id].iloc[0]
    opponent = match_info['opponent']
    stage = match_info['competition_stage']
    
    # Find all Spain goal kicks
    for idx in range(len(match_df)):
        row = match_df.iloc[idx]
        
        # Goal kick detection: type == 'Pass' AND pass_type == 'Goal Kick' AND team == Spain
        if (str(row.get('type', '')).strip() != 'Pass' or
            str(row.get('pass_type', '')).strip() != 'Goal Kick' or
            row.get('team') != TEAM):
            continue
        
        # Find first receiver
        receiver = _find_first_receiver(idx, match_df)
        
        if receiver is not None and receiver['zone'] > 0:
            recv_zone = receiver['zone']
            recv_idx = receiver['iloc']
            recv_player = receiver['player']
            recv_x, recv_y = receiver['x'], receiver['y']
        else:
            # Intercepted → use pass end location
            end_x, end_y = row.get('pass_end_x'), row.get('pass_end_y')
            if pd.notna(end_x) and pd.notna(end_y):
                recv_zone = xy_to_zone(float(end_x), float(end_y))
            else:
                recv_zone = 0
            recv_idx = None
            recv_player = '(intercepted)'
            recv_x, recv_y = end_x, end_y
        
        # Extract chain & outcome
        chain, outcome, granular = _extract_chain_and_outcome(idx, recv_idx, match_df)
        
        # Short/Long classification
        if recv_zone in SHORT_ZONES:
            pass_type = 'short'
        elif recv_zone in LONG_ZONES:
            pass_type = 'long'
        else:
            length = row.get('pass_length')
            pass_type = 'long' if pd.notna(length) and float(length) > 32.0 else 'short'
        
        rec = {
            'match_id': match_id,
            'opponent': opponent,
            'stage': stage,
            'minute': int(row.get('minute', 0)),
            'second': int(row.get('second', 0)),
            'period': int(row.get('period', 1)),
            'gk_player': str(row.get('player', 'GK')),
            'gk_x': row.get('loc_x'),
            'gk_y': row.get('loc_y'),
            'pass_end_x': row.get('pass_end_x'),
            'pass_end_y': row.get('pass_end_y'),
            'pass_height': str(row.get('pass_height', '')),
            'pass_length': row.get('pass_length'),
            'pass_outcome': str(row.get('pass_outcome', 'Complete')),
            'receiver': recv_player,
            'recv_x': recv_x,
            'recv_y': recv_y,
            'recv_zone': recv_zone,
            'pass_type': pass_type,
            'outcome': outcome,
            'granular_outcome': granular,
            'chain': chain,
        }
        all_distributions.append(rec)

# ── Build aggregate data dict (same structure as dashboard) ──
total = len(all_distributions)
short_count = sum(1 for d in all_distributions if d['pass_type'] == 'short')
long_count = total - short_count
short_pct = round(short_count / total * 100, 1) if total else 0.0
long_pct = round(long_count / total * 100, 1) if total else 0.0

zone_counts = {}
zone_outcomes = {}
outcome_counts = {'positive': 0, 'negative': 0}
granular_counts = {'P1': 0, 'P2': 0, 'P3': 0, 'N1': 0, 'N2': 0, 'N3': 0}

for d in all_distributions:
    z = d['recv_zone']
    zone_counts[z] = zone_counts.get(z, 0) + 1
    if z not in zone_outcomes:
        zone_outcomes[z] = {'positive': 0, 'negative': 0}
    zone_outcomes[z][d['outcome']] += 1
    outcome_counts[d['outcome']] += 1
    granular_counts[d['granular_outcome']] = granular_counts.get(d['granular_outcome'], 0) + 1

# Per-match breakdown
match_breakdown = {}
for d in all_distributions:
    key = d['opponent']
    if key not in match_breakdown:
        match_breakdown[key] = {'total': 0, 'short': 0, 'long': 0, 'positive': 0, 'negative': 0}
    match_breakdown[key]['total'] += 1
    match_breakdown[key][d['pass_type']] += 1
    match_breakdown[key][d['outcome']] += 1

print(f'\n🇪🇸 SPAIN — GK BUILD-UP SUMMARY (Euro 2024, all {len(spain_match_ids)} matches)')
print(f'═' * 60)
print(f'   Total Goal Kicks:  {total}')
print(f'   Short:             {short_count} ({short_pct}%)')
print(f'   Long:              {long_count} ({long_pct}%)')
print(f'   Positive outcomes: {outcome_counts["positive"]} ({round(outcome_counts["positive"]/total*100,1) if total else 0}%)')
print(f'   Negative outcomes: {outcome_counts["negative"]} ({round(outcome_counts["negative"]/total*100,1) if total else 0}%)')
print(f'\n   Granular: {granular_counts}')
print(f'\n📊 Per-match breakdown:')
for opp, stats in match_breakdown.items():
    pos_pct = round(stats['positive'] / stats['total'] * 100) if stats['total'] else 0
    print(f"   vs {opp:15s}: {stats['total']} GKs | Short {stats['short']}, Long {stats['long']} | Positive {stats['positive']} ({pos_pct}%)")


🇪🇸 SPAIN — GK BUILD-UP SUMMARY (Euro 2024, all 7 matches)
════════════════════════════════════════════════════════════
   Total Goal Kicks:  51
   Short:             27 (52.9%)
   Long:              24 (47.1%)
   Positive outcomes: 24 (47.1%)
   Negative outcomes: 27 (52.9%)

   Granular: {'P1': 14, 'P2': 10, 'P3': 0, 'N1': 22, 'N2': 5, 'N3': 0}

📊 Per-match breakdown:
   vs England        : 6 GKs | Short 5, Long 1 | Positive 2 (33%)
   vs France         : 11 GKs | Short 7, Long 4 | Positive 7 (64%)
   vs Germany        : 10 GKs | Short 3, Long 7 | Positive 2 (20%)
   vs Georgia        : 4 GKs | Short 4, Long 0 | Positive 4 (100%)
   vs Albania        : 6 GKs | Short 2, Long 4 | Positive 2 (33%)
   vs Italy          : 5 GKs | Short 1, Long 4 | Positive 3 (60%)
   vs Croatia        : 9 GKs | Short 5, Long 4 | Positive 4 (44%)


---
## 2. How We Track Outcome (Possession Chain Logic)

After each goal kick we follow the **event chain**:

1. **Goal Kick** → find the **first receiver** (skip Carry / Pressure events)
2. Track events within a **15-second window** from first reception
3. Use StatsBomb's `possession_team` field to reliably detect possession changes
4. **Classify** the outcome into one of 6 tiers (P1–P3 positive, N1–N3 negative)

> **Key insight:** StatsBomb data includes Pressure and Carry events that don't represent true possession changes — we skip these to avoid false negatives.

---

## 3. Results — Short vs Long Distribution

The first question: **Does Spain prefer to play short or long from goal kicks?**

In [6]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Visualisation: Build-up Type Summary (Short vs Long)
# ══════════════════════════════════════════════════════════════

# ── Colour palette ──
C_SHORT = '#3b82f6'      # blue
C_LONG  = '#f97316'      # orange
C_POS   = '#22c55e'      # green
C_NEG   = '#ef4444'      # red
C_BG    = 'rgba(15,25,35,0.95)'
C_CARD  = 'rgba(25,38,55,0.9)'
C_TEXT  = '#f0f0f0'
C_MUTED = 'rgba(255,255,255,0.5)'

# ── Tournament-level stacked bar ──
fig1a = go.Figure()
fig1a.add_trace(go.Bar(
    y=['All Matches'], x=[short_pct], orientation='h',
    name='Short', marker_color=C_SHORT,
    text=[f'Short {short_count} ({short_pct}%)'],
    textposition='inside', textfont=dict(size=14, color='#fff', family='Arial Black'),
))
fig1a.add_trace(go.Bar(
    y=['All Matches'], x=[long_pct], orientation='h',
    name='Long', marker_color=C_LONG,
    text=[f'Long {long_count} ({long_pct}%)'],
    textposition='inside', textfont=dict(size=14, color='#fff', family='Arial Black'),
))
fig1a.update_layout(
    barmode='stack',
    title=dict(
        text=f'🇪🇸 Spain — GK Distribution Type<br><sup>Euro 2024 · {total} Goal Kicks across 7 matches</sup>',
        font=dict(size=18, color=C_TEXT), x=0.5,
    ),
    plot_bgcolor=C_BG, paper_bgcolor=C_BG, font=dict(color=C_TEXT),
    height=180,
    xaxis=dict(showgrid=False, showticklabels=False, range=[0, 100]),
    yaxis=dict(showgrid=False, showticklabels=False),
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.15, xanchor='center', x=0.5,
                font=dict(size=11)),
    margin=dict(l=10, r=10, t=80, b=10),
)
fig1a.show()

# ── Per-match grouped bar ──
opponents = list(match_breakdown.keys())
short_vals = [match_breakdown[o]['short'] for o in opponents]
long_vals = [match_breakdown[o]['long'] for o in opponents]

fig1b = go.Figure()
fig1b.add_trace(go.Bar(
    x=opponents, y=short_vals, name='Short', marker_color=C_SHORT,
    text=short_vals, textposition='outside', textfont=dict(size=11, color=C_SHORT),
))
fig1b.add_trace(go.Bar(
    x=opponents, y=long_vals, name='Long', marker_color=C_LONG,
    text=long_vals, textposition='outside', textfont=dict(size=11, color=C_LONG),
))
fig1b.update_layout(
    barmode='group',
    title=dict(
        text='Per-Match Distribution Breakdown',
        font=dict(size=14, color=C_TEXT), x=0.5,
    ),
    plot_bgcolor=C_BG, paper_bgcolor=C_BG, font=dict(color=C_TEXT),
    height=350,
    xaxis=dict(tickfont=dict(size=10, color=C_TEXT)),
    yaxis=dict(title=dict(text='Goal Kicks', font=dict(size=10)),
               showgrid=True, gridcolor='rgba(255,255,255,0.08)'),
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                font=dict(size=10)),
    margin=dict(l=40, r=20, t=60, b=40),
)
fig1b.show()

---
## 4. Where Does the Ball Go? — 18-Zone Heatmap

Each zone shows how many times Spain's first receiver collected the ball there, plus the outcome split (🟢 positive / 🔴 negative).

In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Visualisation: 18-Zone Pitch Heatmap
# ══════════════════════════════════════════════════════════════

def pitch_zone_figure_sb(zone_counts, zone_outcomes=None,
                         title='First Receiver Zones', height=480):
    """
    Build a Plotly figure showing the 18-zone pitch heatmap
    using StatsBomb coordinates (120×80).
    """
    fig = go.Figure()
    
    max_count = max(zone_counts.values()) if zone_counts else 1
    
    SB_X_EDGES = [0, 20, 40, 60, 80, 100, 120]
    SB_Y_EDGES = [0, 26.67, 53.33, 80]
    
    for zone_num in range(1, 19):
        row = (zone_num - 1) // 3
        col = (zone_num - 1) % 3
        
        x0 = SB_X_EDGES[row]
        x1 = SB_X_EDGES[row + 1]
        y0 = SB_Y_EDGES[col]
        y1 = SB_Y_EDGES[col + 1]
        
        count = zone_counts.get(zone_num, 0)
        intensity = count / max_count if max_count else 0
        
        # Zone fill: dark navy → warm red based on frequency
        fill_r = int(27 + (200 - 27) * intensity)
        fill_g = int(40 + (40 - 40) * intensity)
        fill_b = int(56 + (56 - 56) * intensity)
        fill_a = 0.25 + 0.6 * intensity
        fill_color = f'rgba({fill_r},{fill_g},{fill_b},{fill_a})'
        
        # Rectangle
        fig.add_shape(
            type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
            line=dict(color='rgba(255,255,255,0.2)', width=1),
            fillcolor=fill_color,
            layer='below',
        )
        
        cx = (x0 + x1) / 2
        cy = (y0 + y1) / 2
        
        if count > 0:
            # Count
            fig.add_annotation(
                x=cx, y=cy + 3,
                text=f'<b>{count}</b>',
                showarrow=False,
                font=dict(size=18, color='#f0f0f0'),
            )
            # Zone label
            fig.add_annotation(
                x=cx, y=cy - 5,
                text=f'Z{zone_num}',
                showarrow=False,
                font=dict(size=9, color='rgba(255,255,255,0.5)'),
            )
            # Outcome dots
            if zone_outcomes and zone_num in zone_outcomes:
                oc = zone_outcomes[zone_num]
                pos_n = oc.get('positive', 0)
                neg_n = oc.get('negative', 0)
                dots = []
                if pos_n > 0:
                    dots.append(f"<span style='color:{C_POS}'>●{pos_n}</span>")
                if neg_n > 0:
                    dots.append(f"<span style='color:{C_NEG}'>●{neg_n}</span>")
                if dots:
                    fig.add_annotation(
                        x=cx, y=cy - 11,
                        text=' '.join(dots),
                        showarrow=False,
                        font=dict(size=9),
                    )
        else:
            fig.add_annotation(
                x=cx, y=cy,
                text=f'Z{zone_num}',
                showarrow=False,
                font=dict(size=9, color='rgba(255,255,255,0.25)'),
            )
    
    # ── Pitch markings ──
    # Halfway line
    fig.add_shape(type='line', x0=60, x1=60, y0=0, y1=80,
                  line=dict(color='rgba(255,255,255,0.2)', width=1, dash='dot'))
    # Centre circle
    import numpy as np
    theta = np.linspace(0, 2*np.pi, 60)
    cx_c, cy_c, r = 60, 40, 9.15
    fig.add_trace(go.Scatter(
        x=cx_c + r * np.cos(theta), y=cy_c + r * np.sin(theta),
        mode='lines', line=dict(color='rgba(255,255,255,0.12)', width=1),
        showlegend=False, hoverinfo='skip',
    ))
    # Penalty areas
    # Left (own)
    fig.add_shape(type='rect', x0=0, x1=18, y0=18, y1=62,
                  line=dict(color='rgba(255,255,255,0.15)', width=1))
    fig.add_shape(type='rect', x0=0, x1=5.5, y0=30, y1=50,
                  line=dict(color='rgba(255,255,255,0.12)', width=1))
    # Right (opponent)
    fig.add_shape(type='rect', x0=102, x1=120, y0=18, y1=62,
                  line=dict(color='rgba(255,255,255,0.15)', width=1))
    fig.add_shape(type='rect', x0=114.5, x1=120, y0=30, y1=50,
                  line=dict(color='rgba(255,255,255,0.12)', width=1))
    
    # Direction arrows
    fig.add_annotation(x=112, y=-4, text='ATK →', showarrow=False,
                       font=dict(size=9, color='rgba(255,255,255,0.4)'))
    fig.add_annotation(x=8, y=-4, text='← OWN GOAL', showarrow=False,
                       font=dict(size=9, color='rgba(255,255,255,0.4)'))
    
    fig.update_layout(
        title=dict(text=title, font=dict(size=14, color='#f0f0f0'), x=0.5),
        xaxis=dict(range=[-3, 123], showgrid=False, zeroline=False,
                   showticklabels=False, fixedrange=True),
        yaxis=dict(range=[-8, 85], showgrid=False, zeroline=False,
                   showticklabels=False, fixedrange=True,
                   scaleanchor='x', scaleratio=0.68),
        plot_bgcolor='rgba(15,25,35,0.85)',
        paper_bgcolor=C_BG,
        margin=dict(l=10, r=10, t=50, b=25),
        height=height,
        showlegend=False,
    )
    return fig


fig2 = pitch_zone_figure_sb(
    zone_counts,
    zone_outcomes,
    title='🇪🇸 Spain — Goal Kick First Receiver Zones<br><sup>Euro 2024 · All Matches · Green=Positive, Red=Negative</sup>',
    height=500,
)
fig2.show()

---
## 5. Outcome Classification — The 6-Tier System

The sunburst shows the full breakdown: inner ring = Positive vs Negative, outer ring = P1–P3 / N1–N3.

In [8]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — Visualisation: Outcome Classification (P1-P3, N1-N3)
# ══════════════════════════════════════════════════════════════

TIER_META = {
    'P1': ('#22c55e', 'Established Possession', 'Kept ball ≥15s without leaving own half'),
    'P2': ('#16a34a', 'Reached Final Third',    'Kept ball ≥15s and reached final third'),
    'P3': ('#15803d', 'Created a Shot',          'Retained possession → shot attempt'),
    'N1': ('#f97316', 'Possession Lost',         'Opponent recovered without entering box'),
    'N2': ('#ef4444', 'Box Entry Conceded',      'Opponent entered penalty area'),
    'N3': ('#dc2626', 'Shot Conceded',           'Opponent produced a shot after recovery'),
}

# ── Sunburst / Treemap of outcomes ──
labels = []
parents = []
values = []
colors = []

# Root
labels.append(f'Total: {total}')
parents.append('')
values.append(total)
colors.append(C_BG)

# Positive / Negative
pos_total = outcome_counts['positive']
neg_total = outcome_counts['negative']

labels.append(f"Positive: {pos_total} ({round(pos_total/total*100)}%)")
parents.append(f'Total: {total}')
values.append(pos_total)
colors.append(C_POS)

labels.append(f"Negative: {neg_total} ({round(neg_total/total*100)}%)")
parents.append(f'Total: {total}')
values.append(neg_total)
colors.append(C_NEG)

# Granular tiers
for code in ['P1', 'P2', 'P3']:
    c = granular_counts.get(code, 0)
    color, label, desc = TIER_META[code]
    labels.append(f'{code}: {c}')
    parents.append(f"Positive: {pos_total} ({round(pos_total/total*100)}%)")
    values.append(c)
    colors.append(color)

for code in ['N1', 'N2', 'N3']:
    c = granular_counts.get(code, 0)
    color, label, desc = TIER_META[code]
    labels.append(f'{code}: {c}')
    parents.append(f"Negative: {neg_total} ({round(neg_total/total*100)}%)")
    values.append(c)
    colors.append(color)

fig3 = go.Figure(go.Sunburst(
    labels=labels,
    parents=parents,
    values=values,
    branchvalues='total',
    marker=dict(colors=colors, line=dict(color=C_BG, width=2)),
    textfont=dict(size=12, color='white'),
    insidetextorientation='radial',
))

fig3.update_layout(
    title=dict(
        text='🇪🇸 Spain — Outcome Classification<br><sup>6-tier system: P1/P2/P3 (positive) · N1/N2/N3 (negative)</sup>',
        font=dict(size=16, color=C_TEXT), x=0.5,
    ),
    plot_bgcolor=C_BG,
    paper_bgcolor=C_BG,
    height=500,
    margin=dict(l=20, r=20, t=80, b=20),
)

fig3.show()

# ── Text summary table ──
print('\n📊 OUTCOME DETAIL')
print('─' * 60)
for code in ['P1', 'P2', 'P3', 'N1', 'N2', 'N3']:
    c = granular_counts.get(code, 0)
    pct = round(c / total * 100, 1) if total else 0
    color_name, label, desc = TIER_META[code]
    marker = '🟢' if code.startswith('P') else '🔴'
    print(f'  {marker} {code} | {c:3d} ({pct:5.1f}%) | {label:25s} | {desc}')


📊 OUTCOME DETAIL
────────────────────────────────────────────────────────────
  🟢 P1 |  14 ( 27.5%) | Established Possession    | Kept ball ≥15s without leaving own half
  🟢 P2 |  10 ( 19.6%) | Reached Final Third       | Kept ball ≥15s and reached final third
  🟢 P3 |   0 (  0.0%) | Created a Shot            | Retained possession → shot attempt
  🔴 N1 |  22 ( 43.1%) | Possession Lost           | Opponent recovered without entering box
  🔴 N2 |   5 (  9.8%) | Box Entry Conceded        | Opponent entered penalty area
  🔴 N3 |   0 (  0.0%) | Shot Conceded             | Opponent produced a shot after recovery


---
## 6. Pass Map — GK to First Receiver

Every line = one goal kick. **Green** = positive, **Red** = negative. Yellow diamond = GK average position.

In [9]:
# ══════════════════════════════════════════════════════════════
# CELL 8 — Visualisation: Pass Map (GK → First Receiver)
# ══════════════════════════════════════════════════════════════

def draw_statsbomb_pitch(fig):
    """Add StatsBomb pitch markings to a figure."""
    # Pitch outline
    fig.add_shape(type='rect', x0=0, x1=120, y0=0, y1=80,
                  line=dict(color='rgba(255,255,255,0.4)', width=2))
    # Halfway line
    fig.add_shape(type='line', x0=60, x1=60, y0=0, y1=80,
                  line=dict(color='rgba(255,255,255,0.3)', width=1))
    # Centre circle
    theta = np.linspace(0, 2*np.pi, 60)
    fig.add_trace(go.Scatter(
        x=60 + 9.15*np.cos(theta), y=40 + 9.15*np.sin(theta),
        mode='lines', line=dict(color='rgba(255,255,255,0.2)', width=1),
        showlegend=False, hoverinfo='skip',
    ))
    # Centre spot
    fig.add_trace(go.Scatter(x=[60], y=[40], mode='markers',
                             marker=dict(size=3, color='rgba(255,255,255,0.3)'),
                             showlegend=False, hoverinfo='skip'))
    # Penalty areas
    fig.add_shape(type='rect', x0=0, x1=18, y0=18, y1=62,
                  line=dict(color='rgba(255,255,255,0.25)', width=1))
    fig.add_shape(type='rect', x0=0, x1=5.5, y0=30, y1=50,
                  line=dict(color='rgba(255,255,255,0.2)', width=1))
    fig.add_shape(type='rect', x0=102, x1=120, y0=18, y1=62,
                  line=dict(color='rgba(255,255,255,0.25)', width=1))
    fig.add_shape(type='rect', x0=114.5, x1=120, y0=30, y1=50,
                  line=dict(color='rgba(255,255,255,0.2)', width=1))
    # Goal areas (6-yard box)
    fig.add_shape(type='rect', x0=0, x1=5.5, y0=30.34, y1=49.66,
                  line=dict(color='rgba(255,255,255,0.15)', width=1))
    fig.add_shape(type='rect', x0=114.5, x1=120, y0=30.34, y1=49.66,
                  line=dict(color='rgba(255,255,255,0.15)', width=1))
    return fig


fig4 = go.Figure()
fig4 = draw_statsbomb_pitch(fig4)

# Plot arrows from GK position to first receiver
for d in all_distributions:
    gx, gy = d['gk_x'], d['gk_y']
    if d['receiver'] == '(intercepted)':
        rx, ry = d['pass_end_x'], d['pass_end_y']
    else:
        rx, ry = d['recv_x'], d['recv_y']
    
    if pd.isna(gx) or pd.isna(gy) or pd.isna(rx) or pd.isna(ry):
        continue
    
    color = C_POS if d['outcome'] == 'positive' else C_NEG
    opacity = 0.7 if d['outcome'] == 'positive' else 0.5
    
    # Line from GK to receiver
    fig4.add_trace(go.Scatter(
        x=[gx, rx], y=[gy, ry],
        mode='lines',
        line=dict(color=color, width=1.5),
        opacity=opacity,
        showlegend=False,
        hoverinfo='text',
        hovertext=f"{d['gk_player']} → {d['receiver']}<br>"
                  f"{d['pass_type'].title()} | {d['outcome'].title()}<br>"
                  f"vs {d['opponent']} ({d['minute']}')" ,
    ))
    
    # Receiver dot
    fig4.add_trace(go.Scatter(
        x=[rx], y=[ry],
        mode='markers',
        marker=dict(size=7, color=color, symbol='circle',
                    line=dict(color='white', width=0.5)),
        opacity=opacity,
        showlegend=False,
        hoverinfo='skip',
    ))

# GK average position
avg_gk_x = np.mean([d['gk_x'] for d in all_distributions if pd.notna(d['gk_x'])])
avg_gk_y = np.mean([d['gk_y'] for d in all_distributions if pd.notna(d['gk_y'])])
fig4.add_trace(go.Scatter(
    x=[avg_gk_x], y=[avg_gk_y],
    mode='markers+text',
    marker=dict(size=14, color='#fbbf24', symbol='diamond',
                line=dict(color='white', width=1.5)),
    text=['GK'], textposition='top center',
    textfont=dict(size=10, color='#fbbf24'),
    showlegend=False, hoverinfo='skip',
))

# Legend proxy traces
fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=8, color=C_POS), name='Positive Outcome', showlegend=True))
fig4.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
    marker=dict(size=8, color=C_NEG), name='Negative Outcome', showlegend=True))

fig4.update_layout(
    title=dict(
        text='🇪🇸 Spain — GK Distribution Pass Map<br><sup>Euro 2024 · Lines from GK to First Receiver · Colour = Outcome</sup>',
        font=dict(size=16, color=C_TEXT), x=0.5,
    ),
    xaxis=dict(range=[-3, 123], showgrid=False, zeroline=False,
               showticklabels=False, fixedrange=True),
    yaxis=dict(range=[-5, 85], showgrid=False, zeroline=False,
               showticklabels=False, fixedrange=True,
               scaleanchor='x', scaleratio=1),
    plot_bgcolor='rgba(15,25,35,0.95)',
    paper_bgcolor=C_BG,
    height=550,
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=-0.05,
                xanchor='center', x=0.5, font=dict(size=10, color=C_TEXT)),
    margin=dict(l=10, r=10, t=70, b=40),
)

fig4.show()

---
## 7. Outcome by Match — Did Opponents Affect Spain's Approach?

Stacked bar showing the 6-tier breakdown per opponent. Which teams pressured Spain most effectively?

In [10]:
# ══════════════════════════════════════════════════════════════
# CELL 9 — Visualisation: Outcome by Match (Stacked Bar)
# ══════════════════════════════════════════════════════════════

# Build per-match granular breakdown
match_granular = {}
for d in all_distributions:
    opp = d['opponent']
    if opp not in match_granular:
        match_granular[opp] = {c: 0 for c in ['P1','P2','P3','N1','N2','N3']}
    match_granular[opp][d['granular_outcome']] += 1

opps = list(match_granular.keys())

fig5 = go.Figure()

for code in ['P3', 'P2', 'P1', 'N1', 'N2', 'N3']:
    color, label, desc = TIER_META[code]
    vals = [match_granular[o].get(code, 0) for o in opps]
    fig5.add_trace(go.Bar(
        x=opps, y=vals,
        name=f'{code} — {label}',
        marker_color=color,
        text=[str(v) if v > 0 else '' for v in vals],
        textposition='inside',
        textfont=dict(size=10, color='white'),
    ))

fig5.update_layout(
    barmode='stack',
    title=dict(
        text='🇪🇸 Spain — Outcome Breakdown by Match<br><sup>6-tier classification per opponent</sup>',
        font=dict(size=16, color=C_TEXT), x=0.5,
    ),
    plot_bgcolor=C_BG,
    paper_bgcolor=C_BG,
    font=dict(color=C_TEXT),
    xaxis=dict(tickfont=dict(size=10)),
    yaxis=dict(title='Goal Kicks', showgrid=True,
               gridcolor='rgba(255,255,255,0.08)'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=9)),
    height=450,
    margin=dict(l=40, r=20, t=80, b=40),
)

fig5.show()

---
## 8. Key Receivers — Who Gets the Ball?

Identifying the **primary first receivers** tells us who to press after a goal kick.

In [11]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — Key Receiver Analysis
# ══════════════════════════════════════════════════════════════

# Who receives Spain's goal kicks?
receiver_stats = {}
for d in all_distributions:
    r = d['receiver']
    if r not in receiver_stats:
        receiver_stats[r] = {'count': 0, 'positive': 0, 'negative': 0, 'zones': []}
    receiver_stats[r]['count'] += 1
    receiver_stats[r][d['outcome']] += 1
    receiver_stats[r]['zones'].append(d['recv_zone'])

recv_df = pd.DataFrame([
    {
        'Receiver': r,
        'Receptions': s['count'],
        'Positive': s['positive'],
        'Negative': s['negative'],
        'Success %': round(s['positive'] / s['count'] * 100, 1) if s['count'] else 0,
        'Avg Zone': round(np.mean(s['zones']), 1) if s['zones'] else 0,
        'Most Freq Zone': f"Z{max(set(s['zones']), key=s['zones'].count)}" if s['zones'] else '-',
    }
    for r, s in receiver_stats.items()
]).sort_values('Receptions', ascending=False)

# Horizontal bar chart
top_receivers = recv_df.head(10)

fig6 = go.Figure()

fig6.add_trace(go.Bar(
    y=top_receivers['Receiver'][::-1],
    x=top_receivers['Positive'][::-1],
    orientation='h',
    name='Positive',
    marker_color=C_POS,
    text=top_receivers['Positive'][::-1],
    textposition='inside',
    textfont=dict(size=10, color='white'),
))

fig6.add_trace(go.Bar(
    y=top_receivers['Receiver'][::-1],
    x=top_receivers['Negative'][::-1],
    orientation='h',
    name='Negative',
    marker_color=C_NEG,
    text=top_receivers['Negative'][::-1],
    textposition='inside',
    textfont=dict(size=10, color='white'),
))

fig6.update_layout(
    barmode='stack',
    title=dict(
        text='🇪🇸 Spain — Top GK Distribution Receivers<br><sup>First receiver after goal kicks · Stacked by outcome</sup>',
        font=dict(size=16, color=C_TEXT), x=0.5,
    ),
    plot_bgcolor=C_BG,
    paper_bgcolor=C_BG,
    font=dict(color=C_TEXT),
    xaxis=dict(title='Receptions', showgrid=True,
               gridcolor='rgba(255,255,255,0.08)'),
    yaxis=dict(tickfont=dict(size=10)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
    height=450,
    margin=dict(l=180, r=20, t=80, b=40),
)

fig6.show()

# Table
print('\n📋 RECEIVER TABLE')
print(recv_df.to_string(index=False))


📋 RECEIVER TABLE
                       Receiver  Receptions  Positive  Negative  Success %  Avg Zone Most Freq Zone
                Aymeric Laporte          10         8         2       80.0       3.4             Z2
            Unai Simón Mendibil           7         2         5       28.6       2.0             Z2
                  (intercepted)           7         0         7        0.0      11.0            Z10
             Mikel Merino Zazón           6         3         3       50.0      10.7            Z10
   Robin Aime Robert Le Normand           5         4         1       80.0       2.8             Z3
     Álvaro Borja Morata Martín           4         2         2       50.0      11.2            Z10
    Lamine Yamal Nasraoui Ebana           3         1         2       33.3      12.7            Z11
José Ignacio Fernández Iglesias           2         1         1       50.0       2.0             Z2
               Fabián Ruiz Peña           2         0         2        0.0       6

---
## 9. Distribution Chains — What Happens After the Goal Kick?

One example chain per outcome tier. 🟢 Green = Spain, 🔴 Red = Opponent. Solid line = pass, dashed = carry.

In [12]:
# ══════════════════════════════════════════════════════════════
# CELL 10b — Distribution Chains (1 example per outcome tier)
# ══════════════════════════════════════════════════════════════
# Replicates the dashboard's chain visualisation.
# Shows 6 chains: one P1, one P2, one P3, one N1, one N2, one N3.
# Chains are capped at MAX_DISPLAY events for readability.

MAX_DISPLAY = 12  # max events to show per chain

# ── Helpers ──────────────────────────────────────────────────
_PASS_EV  = frozenset({'Pass', 'Blocked Pass', 'Offside Pass'})
_SHOT_EV  = frozenset({'Shot', 'Goal', 'Save', 'Saved Shot', 'Miss'})
_SHIELD_EV = frozenset({'Tackle', 'Interception', 'Ball Recovery'})

def _short_label(et: str) -> str:
    return {
        'Pass': 'Pass', 'Ball Receipt*': 'Recv', 'Carry': 'Carry',
        'Ball Recovery': 'Recov.', 'Clearance': 'Clear', 'Aerial Won': 'Aerial',
        'Tackle': 'Tackle', 'Interception': 'Int.', 'Foul Committed': 'Foul',
        'Dispossessed': 'Disp.', 'Miscontrol': 'Misctrl',
        'Shot': 'Shot', 'Goal Keeper': 'GK Act', 'Block': 'Block',
        'Dribble': 'Dribble', 'Duel': 'Duel', 'Error': 'Error',
    }.get(et, et[:8])

def _node_icon(et: str) -> str:
    if et in _PASS_EV:   return '→'
    if et in _SHOT_EV:   return '◎'
    if et in _SHIELD_EV: return '🛡'
    return '●'

# ── Pick one representative example per tier ─────────────────
# Prefer chains with 3-15 events (readable length); fallback to any.
tier_order = ['P1', 'P2', 'P3', 'N1', 'N2', 'N3']
tier_examples = {}

for d in all_distributions:
    g = d['granular_outcome']
    clen = len(d.get('chain', []))
    if g not in tier_examples:
        tier_examples[g] = d
    else:
        prev_len = len(tier_examples[g].get('chain', []))
        # Prefer chains in the 3-15 "sweet spot"; among those pick longest
        prev_ok = 3 <= prev_len <= 15
        curr_ok = 3 <= clen <= 15
        if curr_ok and (not prev_ok or clen > prev_len):
            tier_examples[g] = d
        elif not prev_ok and not curr_ok and clen > prev_len:
            tier_examples[g] = d

# ── Build one Plotly figure per chain ────────────────────────
TIER_COLORS = {
    'P1': '#22c55e', 'P2': '#16a34a', 'P3': '#15803d',
    'N1': '#f97316', 'N2': '#ef4444', 'N3': '#dc2626',
}

for tier in tier_order:
    if tier not in tier_examples:
        continue
    d = tier_examples[tier]
    full_chain = d.get('chain', [])
    if not full_chain:
        continue

    tier_color = TIER_COLORS[tier]
    _, tier_label, tier_desc = TIER_META[tier]

    # Truncate chain for display
    truncated = len(full_chain) > MAX_DISPLAY
    chain = full_chain[:MAX_DISPLAY]
    n = len(chain)
    extra = len(full_chain) - MAX_DISPLAY if truncated else 0

    node_spacing = 1.0
    total_nodes = n + (1 if truncated else 0)  # extra slot for "..." node

    fig = go.Figure()

    # ── Connectors (lines between nodes) ──
    for i in range(n - 1):
        evt_next = chain[i + 1]
        et_next = evt_next.get('event_type', '')
        is_pass = et_next in _PASS_EV
        x0, x1 = i * node_spacing + 0.3, (i + 1) * node_spacing - 0.3

        fig.add_trace(go.Scatter(
            x=[x0, x1], y=[0, 0], mode='lines',
            line=dict(
                color='rgba(255,255,255,0.3)' if is_pass else 'rgba(255,255,255,0.15)',
                width=2 if is_pass else 1.5,
                dash='solid' if is_pass else 'dash',
            ),
            showlegend=False, hoverinfo='skip',
        ))
        fig.add_annotation(
            x=x1, y=0, ax=x1 - 0.15, ay=0,
            xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,
            arrowcolor='rgba(255,255,255,0.25)',
        )

    # ── Nodes ──
    for i, evt in enumerate(chain):
        is_team = evt.get('is_team', True)
        et = evt.get('event_type', '')
        player = evt.get('player', '?')

        nc = '#22c55e' if is_team else '#ef4444'
        nb = 'rgba(34,197,94,0.15)' if is_team else 'rgba(239,68,68,0.15)'
        tc = '#d1fae5' if is_team else '#fecaca'

        fig.add_trace(go.Scatter(
            x=[i * node_spacing], y=[0], mode='markers+text',
            marker=dict(size=32, color=nb, line=dict(color=nc, width=2)),
            text=[_node_icon(et)], textfont=dict(size=12, color=nc),
            textposition='middle center', showlegend=False, hoverinfo='text',
            hovertext=f"<b>{player}</b><br>{et}<br>{'Spain' if is_team else 'Opponent'}",
        ))
        short_name = player.split()[-1][:12] if player != '?' else '?'
        fig.add_annotation(x=i * node_spacing, y=0.55,
                           text=f"<b>{short_name}</b>", showarrow=False,
                           font=dict(size=9, color=tc))
        fig.add_annotation(x=i * node_spacing, y=-0.55,
                           text=_short_label(et), showarrow=False,
                           font=dict(size=8, color='rgba(255,255,255,0.5)'))

    # ── Truncation "..." node ──
    if truncated:
        trunc_x = n * node_spacing
        # Dashed connector to "..."
        fig.add_trace(go.Scatter(
            x=[(n - 1) * node_spacing + 0.3, trunc_x - 0.3], y=[0, 0],
            mode='lines', line=dict(color='rgba(255,255,255,0.12)', width=1.5, dash='dot'),
            showlegend=False, hoverinfo='skip',
        ))
        fig.add_trace(go.Scatter(
            x=[trunc_x], y=[0], mode='markers+text',
            marker=dict(size=28, color='rgba(255,255,255,0.06)',
                        line=dict(color='rgba(255,255,255,0.2)', width=1.5)),
            text=['⋯'], textfont=dict(size=16, color='rgba(255,255,255,0.5)'),
            textposition='middle center', showlegend=False, hoverinfo='text',
            hovertext=f"+{extra} more events",
        ))
        fig.add_annotation(x=trunc_x, y=-0.55,
                           text=f"+{extra} more", showarrow=False,
                           font=dict(size=8, color='rgba(255,255,255,0.35)'))

    # ── Header ──
    gk_name = d.get('gk_player', 'GK')
    minute = d.get('minute', 0)
    second = d.get('second', 0)
    pass_type = d.get('pass_type', 'short')
    opponent = d.get('opponent', '?')
    pt_color = C_SHORT if pass_type == 'short' else C_LONG
    chain_len = len(full_chain)

    title_text = (
        f"<span style='color:{tier_color};font-weight:700'>{tier}</span>"
        f" — {tier_label}"
        f"<br><sup style='color:rgba(255,255,255,0.6)'>"
        f"{gk_name} · {minute}'{second:02d}\" · "
        f"<span style='color:{pt_color}'>{pass_type.upper()}</span>"
        f" · vs {opponent} · {chain_len} events</sup>"
    )

    x_max = (total_nodes - 1) * node_spacing + 0.7
    fig.update_layout(
        title=dict(text=title_text, font=dict(size=13, color=C_TEXT), x=0.01, xanchor='left'),
        xaxis=dict(range=[-0.7, x_max], showgrid=False, zeroline=False,
                   showticklabels=False, fixedrange=True),
        yaxis=dict(range=[-1.3, 1.3], showgrid=False, zeroline=False,
                   showticklabels=False, fixedrange=True),
        plot_bgcolor=C_CARD, paper_bgcolor=C_BG,
        height=155, margin=dict(l=10, r=10, t=50, b=10),
        showlegend=False,
    )
    fig.show()

# ── Summary ──
print(f'\n📋 Showing 1 example chain per outcome tier ({len(tier_examples)} tiers found)')
print(f'   Display capped at {MAX_DISPLAY} events (⋯ = truncated)\n')
for t in tier_order:
    if t in tier_examples:
        d = tier_examples[t]
        clen = len(d.get('chain', []))
        trunc = ' (truncated)' if clen > MAX_DISPLAY else ''
        print(f'   {t}: vs {d["opponent"]} {d["minute"]}\'{d["second"]:02d}" '
              f'({d["pass_type"]}, {clen} events{trunc})')


📋 Showing 1 example chain per outcome tier (4 tiers found)
   Display capped at 12 events (⋯ = truncated)

   P1: vs England 0'34" (short, 14 events (truncated))
   P2: vs Italy 25'46" (short, 12 events)
   N1: vs Germany 60'57" (short, 13 events (truncated))
   N2: vs Croatia 22'20" (short, 10 events)


---
## 10. Opposition Intelligence — Tactical Recommendations for Italy

Bringing everything together into actionable coaching insights.

In [13]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — Opposition Intelligence Summary
# ══════════════════════════════════════════════════════════════

# Compute summary metrics
pos_rate = round(outcome_counts['positive'] / total * 100, 1) if total else 0
neg_rate = round(outcome_counts['negative'] / total * 100, 1) if total else 0

# Most targeted zones
sorted_zones = sorted(zone_counts.items(), key=lambda x: x[1], reverse=True)
top_3_zones = sorted_zones[:3]

# Short vs Long by half
h1_short = sum(1 for d in all_distributions if d['period'] == 1 and d['pass_type'] == 'short')
h1_long = sum(1 for d in all_distributions if d['period'] == 1 and d['pass_type'] == 'long')
h2_short = sum(1 for d in all_distributions if d['period'] == 2 and d['pass_type'] == 'short')
h2_long = sum(1 for d in all_distributions if d['period'] == 2 and d['pass_type'] == 'long')
et_short = sum(1 for d in all_distributions if d['period'] > 2 and d['pass_type'] == 'short')
et_long = sum(1 for d in all_distributions if d['period'] > 2 and d['pass_type'] == 'long')

# Top receiver
top_recv = recv_df.iloc[0]

# Most vulnerable zone (highest negative rate)
vuln_zones = []
for z, oc in zone_outcomes.items():
    total_z = oc['positive'] + oc['negative']
    if total_z >= 2:
        neg_rate_z = oc['negative'] / total_z * 100
        vuln_zones.append((z, neg_rate_z, total_z))
vuln_zones.sort(key=lambda x: x[1], reverse=True)

# GK who takes goal kicks
gk_players = {}
for d in all_distributions:
    gk = d['gk_player']
    gk_players[gk] = gk_players.get(gk, 0) + 1

print('═' * 70)
print('🇪🇸  SPAIN — GK BUILD-UP OPPOSITION INTELLIGENCE')
print('   UEFA Euro 2024 — Prepared for Italy Coaching Staff')
print('═' * 70)

print(f'\n📊 OVERVIEW')
print(f'   Total Goal Kicks analysed: {total} (across {len(spain_match_ids)} matches)')
print(f'   GK taking kicks: {dict(sorted(gk_players.items(), key=lambda x: -x[1]))}')

print(f'\n🔵 DISTRIBUTION PATTERN')
print(f'   Short: {short_count} ({short_pct}%) | Long: {long_count} ({long_pct}%)')
print(f'   → Spain STRONGLY prefers short build-up from goal kicks')
print(f'   1st Half: Short {h1_short}, Long {h1_long}')
print(f'   2nd Half: Short {h2_short}, Long {h2_long}')
if et_short + et_long > 0:
    print(f'   Extra Time: Short {et_short}, Long {et_long}')

print(f'\n🎯 TARGET ZONES (Top 3)')
for z, c in top_3_zones:
    oc = zone_outcomes.get(z, {})
    pos = oc.get('positive', 0)
    neg = oc.get('negative', 0)
    print(f'   Z{z}: {c} distributions ({pos}✅ {neg}❌)')

print(f'\n👤 TOP RECEIVER')
print(f'   {top_recv["Receiver"]}: {top_recv["Receptions"]} receptions, '
      f'{top_recv["Success %"]}% success, most often in {top_recv["Most Freq Zone"]}')

print(f'\n✅ POSSESSION RETENTION: {pos_rate}%')
print(f'   P1 (Established): {granular_counts["P1"]}')
print(f'   P2 (Final Third): {granular_counts["P2"]}')
print(f'   P3 (Shot Created): {granular_counts["P3"]}')

print(f'\n❌ POSSESSION LOSS: {neg_rate}%')
print(f'   N1 (Lost - no danger): {granular_counts["N1"]}')
print(f'   N2 (Box entry conceded): {granular_counts["N2"]}')
print(f'   N3 (Shot conceded): {granular_counts["N3"]}')

if vuln_zones:
    print(f'\n⚠️  VULNERABLE ZONES (highest negative rate, min 2 distributions):')
    for z, neg_r, tot in vuln_zones[:3]:
        print(f'   Z{z}: {neg_r:.0f}% negative ({tot} total)')

print(f'\n💡 TACTICAL RECOMMENDATIONS FOR ITALY:')
print(f'   1. HIGH PRESS on short goal kicks — Spain plays short {short_pct}% of the time')
print(f'   2. Target Z{top_3_zones[0][0]} and Z{top_3_zones[1][0]} — most frequent receiving zones')
print(f'   3. Press {top_recv["Receiver"]} — primary first receiver ({top_recv["Receptions"]} receptions)')
if vuln_zones:
    print(f'   4. Force play toward Z{vuln_zones[0][0]} — highest negative outcome rate ({vuln_zones[0][1]:.0f}%)')

══════════════════════════════════════════════════════════════════════
🇪🇸  SPAIN — GK BUILD-UP OPPOSITION INTELLIGENCE
   UEFA Euro 2024 — Prepared for Italy Coaching Staff
══════════════════════════════════════════════════════════════════════

📊 OVERVIEW
   Total Goal Kicks analysed: 51 (across 7 matches)
   GK taking kicks: {'Unai Simón Mendibil': 35, 'Aymeric Laporte': 8, 'David Raya Martin': 5, 'José Ignacio Fernández Iglesias': 3}

🔵 DISTRIBUTION PATTERN
   Short: 27 (52.9%) | Long: 24 (47.1%)
   → Spain STRONGLY prefers short build-up from goal kicks
   1st Half: Short 13, Long 5
   2nd Half: Short 14, Long 15
   Extra Time: Short 0, Long 4

🎯 TARGET ZONES (Top 3)
   Z2: 16 distributions (7✅ 9❌)
   Z10: 10 distributions (3✅ 7❌)
   Z12: 6 distributions (2✅ 4❌)

👤 TOP RECEIVER
   Aymeric Laporte: 10 receptions, 80.0% success, most often in Z2

✅ POSSESSION RETENTION: 47.1%
   P1 (Established): 14
   P2 (Final Third): 10
   P3 (Shot Created): 0

❌ POSSESSION LOSS: 52.9%
   N1 (Lost 